# Support Vector Machines (Interview Level) — AI Lab Instructor Notebook

*Classical ML · Medium*

Build strong intuition for max-margin classification, kernels, and when SVMs fail.

**Outcome.** Students can:
- explain hard vs soft margin,
- understand hinge loss and margin,
- train LinearSVC/SVC in sklearn,
- reason about scaling, C, kernels, and failure modes.

5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# SVM (Interview Level) — Student Lab

We use sklearn here, but you still compute hinge loss and reason about margins.

## Section 0 — Synthetic datasets
We create 2D datasets for intuition.

### Figure: Linearly Separable Dataset
![Linear dataset for SVM](/images/w4-d2-linear.png)

### Figure: XOR Dataset (not linearly separable)
![XOR dataset for SVM](/images/w4-d2-xor.png)

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_linear(n=300, sep=2.0):
    X0 = rng.standard_normal((n//2, 2)) + np.array([0,0])
    X1 = rng.standard_normal((n-n//2, 2)) + np.array([sep,0])
    X = np.vstack([X0, X1])
    y = np.array([0]*len(X0) + [1]*len(X1))
    p = rng.permutation(n)
    return X[p], y[p]

def make_xor(n=400, noise=0.2):
    X = rng.uniform(-1, 1, size=(n,2))
    y = (X[:,0]*X[:,1] > 0).astype(int)
    flip = rng.random(n) < noise
    y[flip] = 1 - y[flip]
    return X, y

X_lin, y_lin = make_linear()
X_xor, y_xor = make_xor()

## Task 2: Implement hinge loss for y in {-1,+1}
*Section: Hinge loss*

## Section 1 — Hinge loss

### Task 1.1: Implement hinge loss for y in {-1,+1}

L = mean(max(0, 1 - y*(Xw + b)))

**Hints**
- convert y from {0,1} to {-1,+1} as y2 = 2y-1

In [ ]:
def hinge_loss(X, y_pm1, w, b):
  #Math: vectors are columns → write -> 𝑤tx
  #Code: vectors are 1D → write x @ w
    margins = y_pm1 * (X @ w + b)
    return float(np.mean(np.maximum(0.0, 1.0 - margins)))

X = np.array([[1.0, 0.0], [0.0, 1.0]])
y = np.array([1, 0])
y_pm1 = 2*y - 1
w = np.array([1.0, -1.0]); b = 0.0
L = hinge_loss(X, y_pm1, w, b)
print('hinge', L)
# Rationale: hinge penalizes points inside margin; encourages maximum margin.

In [ ]:
# Checks
check('hinge_numeric', isinstance(L, (int, float, np.integer, np.floating)))

**Why this works.** hinge penalizes points inside margin; encourages maximum margin.

## Task 3: Standardize + fit for different C
*Section: Linear SVM (LinearSVC)*

## Section 2 — Linear SVM (LinearSVC)

### Task 2.1: Standardize + fit for different C

![SVM margin and support vectors](/images/w4-d2-svm-margin-support-vectors.png)

Use a train/validation split. On easy data, train accuracy can look flat across C; val accuracy and margin behavior are more informative.

**FAANG gotcha:** Without scaling, SVM behaves poorly.

In [ ]:
$10

In [ ]:
# Checks
scaler_t, clf_t = fit_linear_svm(Xtr, ytr_noisy, C=1.0)
acc_t = clf_t.score(scaler_t.transform(Xva), yva)
check('acc_finite', np.isfinite(acc_t))

**Why this works.** C trades margin width vs training errors.

## Task 4: XOR dataset
*Section: Kernel SVM (RBF)*

## Section 3 — Kernel SVM (RBF)

### Task 3.1: XOR dataset

![RBF kernel trick](/images/w4-d2-rbf-kernel-trick.png)

Train:
- linear kernel (should struggle)
- RBF kernel (should improve)

**C controls how much SVM cares about training errors.**

- low C -> underfit
- medium C -> best
- high C -> overfit

**Think of C as strictness of a teacher grading exams:**

- C = 0.01: Chill teacher - small mistakes OK
- C = 1: Balanced teacher
- C = 100: Angry teacher - zero tolerance

In [ ]:
scaler = StandardScaler()
Xx = scaler.fit_transform(X_xor)

clf_lin = SVC(kernel='linear', C=1.0)
clf_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
# RBF = Radial Basis Function kernel
clf_lin.fit(Xx, y_xor)
clf_rbf.fit(Xx, y_xor)
print('linear acc', clf_lin.score(Xx, y_xor))
print('rbf acc', clf_rbf.score(Xx, y_xor))
# Rationale: RBF kernel implicitly maps to higher-dim feature space enabling nonlinear boundary.

**Why this works.** RBF kernel implicitly maps to higher-dim feature space enabling nonlinear boundary.

## Task 5: Show scaling sensitivity
*Section: Failure modes*

## Section 4 — Failure modes

### Task 4.1: Show scaling sensitivity
Multiply one feature by 1000 and compare performance with/without StandardScaler.

In [ ]:
X_bad = X_lin.copy()
X_bad[:, 1] *= 1000

clf = LinearSVC(C=1.0, max_iter=20000, random_state=0)
clf.fit(X_bad, y_lin)
acc_noscale = clf.score(X_bad, y_lin)

scaler = StandardScaler()
Xs = scaler.fit_transform(X_bad)
clf2 = LinearSVC(C=1.0, max_iter=20000, random_state=0)
clf2.fit(Xs, y_lin)
acc_scale = clf2.score(Xs, y_lin)

print('acc_noscale', acc_noscale, 'acc_scale', acc_scale)
# Rationale: without scaling, a single large-scale feature dominates distances/margins.

In [ ]:
# Checks
check('scaling_helpful', acc_scale >= acc_noscale - 1e-9)

**Why this works.** without scaling, a single large-scale feature dominates distances/margins.